In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/Colab Notebooks/AI/FruitinAmazon.zip"
extract_path = "FruitinAmazon"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Done extracting!")

Done extracting!


In [ ]:
os.listdir("/content/FruitinAmazon/FruitinAmazon/test")


['graviola', 'guarana', 'acai', 'pupunha', 'cupuacu', 'tucuma']

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_data = train_datagen.flow_from_directory(
    "/content/FruitinAmazon/FruitinAmazon/train",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

val_data = train_datagen.flow_from_directory(
    "/content/FruitinAmazon/FruitinAmazon/train",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

test_datagen = ImageDataGenerator(rescale=1./255)

test_data = test_datagen.flow_from_directory(
    "/content/FruitinAmazon/FruitinAmazon/test",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

Found 72 images belonging to 6 classes.
Found 18 images belonging to 6 classes.
Found 30 images belonging to 6 classes.


In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze layers
for layer in base_model.layers:
    layer.trainable = False

# Custom head
x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.5)(x)
output = layers.Dense(train_data.num_classes, activation='softmax')(x)

model = models.Model(inputs=base_model.input, outputs=output)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

Epoch 1/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 12s 2s/step - accuracy: 0.2500 - loss: 2.2865 - val_accuracy: 0.4444 - val_loss: 1.3642
Epoch 2/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 963ms/step - accuracy: 0.4028 - loss: 1.2885 - val_accuracy: 0.4444 - val_loss: 1.0783
Epoch 3/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 5s 2s/step - accuracy: 0.7361 - loss: 0.7940 - val_accuracy: 0.6111 - val_loss: 0.8253
Epoch 4/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 923ms/step - accuracy: 0.8333 - loss: 0.6085 - val_accuracy: 0.8333 - val_loss: 0.6406
Epoch 5/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 944ms/step - accuracy: 0.8750 - loss: 0.4516 - val_accuracy: 0.8889 - val_loss: 0.5240
Epoch 6/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 1s/step - accuracy: 0.9444 - loss: 0.2768 - val_accuracy: 0.8333 - val_loss: 0.4731
Epoch 7/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 2s/step - accuracy: 0.9444 - loss: 0.2410 - val_accuracy: 0.7778 - val_loss: 0.4782
Epoch 8/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 1s/step - accuracy: 0.9583 - loss: 0.1829 - val_accuracy: 0.8333 - val_loss: 0.4893
Epoch 

In [ ]:
loss, acc = model.evaluate(test_data)
print("Test Accuracy:", acc)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.9000 - loss: 0.3198
Test Accuracy: 0.8999999761581421


In [ ]:
from sklearn.metrics import classification_report
import numpy as np

predictions = model.predict(test_data)
y_pred = np.argmax(predictions, axis=1)

print(classification_report(
    test_data.classes,
    y_pred,
    target_names=list(test_data.class_indices.keys())
))

1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step
              precision    recall  f1-score   support

        acai       1.00      1.00      1.00         5
     cupuacu       1.00      0.80      0.89         5
    graviola       1.00      1.00      1.00         5
     guarana       1.00      1.00      1.00         5
     pupunha       0.80      0.80      0.80         5
      tucuma       0.67      0.80      0.73         5

    accuracy                           0.90        30
   macro avg       0.91      0.90      0.90        30
weighted avg       0.91      0.90      0.90        30



In [ ]:
from tensorflow.keras.preprocessing import image
import numpy as np

img_path = "/content/FruitinAmazon/FruitinAmazon/train/tucuma/images (1).jpeg"  # change this

img = image.load_img(img_path, target_size=(224, 224))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

pred = model.predict(img_array)
pred_class = list(train_data.class_indices.keys())[np.argmax(pred)]

print("Prediction:", pred_class)

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
Prediction: tucuma


In [ ]:
from tensorflow.keras import Sequential

scratch_model = Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)),
    layers.MaxPooling2D(),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(train_data.num_classes, activation='softmax')
])

scratch_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

scratch_model.fit(train_data, validation_data=val_data, epochs=10)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 10s 3s/step - accuracy: 0.1111 - loss: 12.7118 - val_accuracy: 0.1667 - val_loss: 10.7413
Epoch 2/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 6s 2s/step - accuracy: 0.1111 - loss: 7.8339 - val_accuracy: 0.3333 - val_loss: 2.2557
Epoch 3/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 8s 3s/step - accuracy: 0.4444 - loss: 1.7853 - val_accuracy: 0.3333 - val_loss: 1.6500
Epoch 4/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 6s 2s/step - accuracy: 0.4306 - loss: 1.3756 - val_accuracy: 0.3889 - val_loss: 1.4568
Epoch 5/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 7s 2s/step - accuracy: 0.6667 - loss: 1.1180 - val_accuracy: 0.2778 - val_loss: 1.5274
Epoch 6/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 8s 3s/step - accuracy: 0.7500 - loss: 0.7728 - val_accuracy: 0.3333 - val_loss: 1.2789
Epoch 7/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 7s 2s/step - accuracy: 0.9028 - loss: 0.4671 - val_accuracy: 0.5000 - val_loss: 1.3015
Epoch 8/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 9s 2s/step - accuracy: 0.8889 - loss: 0.3105 - val_accuracy: 0.4444 - val_loss: 1.6618
Epoch 9/10
3/

In [ ]:
predictions = scratch_model.predict(test_data)
y_pred = np.argmax(predictions, axis=1)

print(classification_report(
    test_data.classes,
    y_pred,
    target_names=list(test_data.class_indices.keys())
))

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 658ms/step
              precision    recall  f1-score   support

        acai       0.71      1.00      0.83         5
     cupuacu       0.67      0.40      0.50         5
    graviola       0.56      1.00      0.71         5
     guarana       0.57      0.80      0.67         5
     pupunha       1.00      0.20      0.33         5
      tucuma       0.67      0.40      0.50         5

    accuracy                           0.63        30
   macro avg       0.70      0.63      0.59        30
weighted avg       0.70      0.63      0.59        30



In [ ]:
from tensorflow.keras.preprocessing import image
import numpy as np

img_path = "/content/FruitinAmazon/FruitinAmazon/train/tucuma/images (2).jpeg"  # change this

img = image.load_img(img_path, target_size=(224, 224))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

pred = scratch_model.predict(img_array)
pred_class = list(train_data.class_indices.keys())[np.argmax(pred)]

print("Prediction:", pred_class)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 658ms/step
Prediction: tucuma
